In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# ============================================================
# FINAL MULTIMODAL PIPELINE — PROJECT SETUP
# ============================================================


In [2]:

import sys
import json
import importlib
from pathlib import Path

from PIL import Image
from google.colab import userdata
from google import genai

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project"
)

CODE_DIR = PROJECT_ROOT / "03_Code"
MODEL_DIR = PROJECT_ROOT / "05_Models"
RESULTS_DIR = PROJECT_ROOT / "04_Results" / "Multimodal"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("Project root:", PROJECT_ROOT)
print("Code directory:", CODE_DIR)

Project root: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project
Code directory: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/03_Code


# ============================================================
# IMPORT EXISTING PROJECT MODULES
# ============================================================

In [3]:


from EEG.eeg_inference import infer_eeg
from EMG.emg_inference import infer_emg

from Vision.vision_inference import infer_vision
from Vision.vision_model import create_vision_client

from Prompt.mappings import (
    EEG_GESTURE_MAP,
    EMG_LABEL_MAP,
)

from Prompt.normalization import (
    normalize_eeg_output,
    normalize_emg_output,
)

from Prompt.context import (
    build_vision_context,
    build_multimodal_context,
)

from Prompt.prompt_builder import build_reasoning_prompt

from Prompt.llm_reasoner import (
    create_gemini_client,
    run_reasoning,
    DEFAULT_MODEL,
)

from Prompt.fusion import deterministic_fusion

print("All project modules imported successfully.")

All project modules imported successfully.


# ============================================================
# MODEL AND CLIENT INITIALIZATION
# ============================================================

In [4]:


API_KEY = userdata.get("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "API_KEY was not found in Colab Secrets."
    )

# Separate clients for Vision and LLM reasoning
vision_client = create_vision_client(API_KEY)
reasoning_client = create_gemini_client(API_KEY)

print("Vision client initialized.")
print("Reasoning client initialized.")
print("Reasoning model:", DEFAULT_MODEL)

Vision client initialized.
Reasoning client initialized.
Reasoning model: gemini-3.1-flash-lite


# ============================================================
# LIVE VISION INFERENCE
# ============================================================


In [5]:

def run_live_vision(image_path):
    """
    Run Vision inference directly on the provided image.

    Parameters
    ----------
    image_path : str or Path
        Path to the input image.

    Returns
    -------
    dict
        Standardized Vision output.
    """

    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    image = Image.open(image_path).convert("RGB")

    vision_output = infer_vision(
        client=vision_client,
        image=image,
    )

    return vision_output




# ============================================================
# LIVE MULTIMODAL PIPELINE
# ============================================================

In [6]:


def run_multimodal_pipeline(
    task_instruction,
    image_path=None,
    eeg_path=None,
    emg_path=None,
    run_llm=True,
):
    """
    Run selected modalities and construct multimodal reasoning.

    Parameters
    ----------
    task_instruction : str
        User-provided task or action instruction.

    image_path : str or Path, optional
        Input image path.

    eeg_path : str or Path, optional
        Input EEG file path.

    emg_path : str or Path, optional
        Input EMG file path.

    run_llm : bool
        Whether to run Gemini reasoning.

    Returns
    -------
    dict
        Complete multimodal result.
    """

    if not any([
        image_path is not None,
        eeg_path is not None,
        emg_path is not None,
    ]):
        raise ValueError(
            "At least one modality must be provided."
        )

    # --------------------------------------------------------
    # EEG
    # --------------------------------------------------------

    if eeg_path is not None:

        eeg_raw_output = infer_eeg(
            file_path=str(eeg_path),
            model=eeg_model,
            device=DEVICE,
        )

        eeg_output = normalize_eeg_output(
            raw_label=f"G{eeg_raw_output['class_id'] + 1}",
            confidence=eeg_raw_output["confidence"],
        )

    else:
        eeg_output = {}

    # --------------------------------------------------------
    # EMG
    # --------------------------------------------------------

    if emg_path is not None:

        emg_raw_output = infer_emg(
            file_path=str(emg_path),
            model=emg_model,
            device=DEVICE,
        )

        emg_output = normalize_emg_output(
            raw_label=emg_raw_output["class_id"],
            confidence=emg_raw_output["confidence"],
        )

    else:
        emg_output = {}

    # --------------------------------------------------------
    # Vision
    # --------------------------------------------------------

    if image_path is not None:

        vision_raw_output = run_live_vision(
            image_path=image_path
        )

        vision_output = build_vision_context(
            hands=vision_raw_output.get("hands", [])
        )

    else:
        vision_output = {}

    # --------------------------------------------------------
    # Multimodal context
    # --------------------------------------------------------

    multimodal_context = build_multimodal_context(
        task_instruction=task_instruction,
        eeg_output=eeg_output,
        emg_output=emg_output,
        vision_output=vision_output,
    )

    # --------------------------------------------------------
    # Deterministic baseline
    # --------------------------------------------------------

    baseline_result = deterministic_fusion(
        multimodal_context
    )

    result = {
        "task_instruction": task_instruction,
        "eeg_output": eeg_output,
        "emg_output": emg_output,
        "vision_output": vision_output,
        "multimodal_context": multimodal_context,
        "deterministic_result": baseline_result,
    }

    # --------------------------------------------------------
    # LLM reasoning
    # --------------------------------------------------------

    if run_llm:

        reasoning_prompt = build_reasoning_prompt(
            multimodal_context
        )

        llm_result = run_reasoning(
            client=reasoning_client,
            prompt=reasoning_prompt,
            model=DEFAULT_MODEL,
        )

        result["reasoning_prompt"] = reasoning_prompt
        result["llm_result"] = llm_result

    return result



import json
def display_clean_result(result):
    """Display the important multimodal inference results."""

    print("=" * 70)
    print("MULTIMODAL INFERENCE RESULT")
    print("=" * 70)

    print("\nTASK:")
    print(result.get("task_instruction"))

    print("\nEEG:")
    print(json.dumps(result.get("eeg_output"), indent=2))

    print("\nEMG:")
    print(json.dumps(result.get("emg_output"), indent=2))

    print("\nVISION:")
    print(json.dumps(result.get("vision_output"), indent=2))

    print("\nDETERMINISTIC FUSION:")
    print(json.dumps(result.get("deterministic_result"), indent=2))

    print("\nLLM REASONING:")
    llm_result = result.get("llm_result")

    if hasattr(llm_result, "parsed") and llm_result.parsed:
        parsed = llm_result.parsed
    elif isinstance(llm_result, dict):
        parsed = llm_result
    else:
        parsed = {"raw_output": str(llm_result)}

    print(json.dumps(parsed, indent=2))

    print("=" * 70)

In [7]:
import os
import torch

# --------------------------------------------------
# 1. Device
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# --------------------------------------------------
# 2. Import model architectures
# --------------------------------------------------

from EEG.eeg_decoder import EEGDecoder
from EMG.emg_model import EMGNet


# --------------------------------------------------
# 3. Model paths
# --------------------------------------------------

EEG_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "05_Models",
    "EEG",
    "eeg_decoder_best_model.pth"
)

EMG_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "05_Models",
    "EMG",
    "emgnet_model.pth"
)

print("EEG model exists:", os.path.exists(EEG_MODEL_PATH))
print("EMG model exists:", os.path.exists(EMG_MODEL_PATH))


# --------------------------------------------------
# 4. Initialize EEG model
# --------------------------------------------------

eeg_model = EEGDecoder(
    num_classes=7
)

eeg_checkpoint = torch.load(
    EEG_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

if isinstance(eeg_checkpoint, dict) and "model_state_dict" in eeg_checkpoint:
    eeg_model.load_state_dict(eeg_checkpoint["model_state_dict"])
else:
    eeg_model.load_state_dict(eeg_checkpoint)

eeg_model = eeg_model.to(DEVICE)
eeg_model.eval()


# --------------------------------------------------
# 5. Initialize EMG model
# --------------------------------------------------

emg_model = EMGNet(
    out_gesture=4
)

emg_checkpoint = torch.load(
    EMG_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

if isinstance(emg_checkpoint, dict) and "model_state_dict" in emg_checkpoint:
    emg_model.load_state_dict(emg_checkpoint["model_state_dict"])
else:
    emg_model.load_state_dict(emg_checkpoint)

emg_model = emg_model.to(DEVICE)
emg_model.eval()


print("✅ EEG model loaded")
print("✅ EMG model loaded")

Using device: cpu
EEG model exists: True
EMG model exists: True
✅ EEG model loaded
✅ EMG model loaded


In [8]:
# ============================================================
# END-TO-END MULTIMODAL TEST
# ============================================================

IMAGE_PATH = (
    PROJECT_ROOT
    / "01_Data/Image/hands23/jpg/AR__4R6QekTdrw_9_2070_15.jpg"
)

EEG_PATH = (
    PROJECT_ROOT
    / "01_Data/EEG/raw/mat_data/subject_1/S1_R1_G1.mat"
)

EMG_PATH = (
    PROJECT_ROOT
    / "01_Data/EMG/raw/subject_22/S22_R1_G1.mat"
)

result = run_multimodal_pipeline(
    task_instruction="Pick up the cup.",
    image_path=IMAGE_PATH,
    eeg_path=EEG_PATH,
    emg_path=EMG_PATH,
    run_llm=True,
)

display_clean_result(result)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


MULTIMODAL INFERENCE RESULT

TASK:
Pick up the cup.

EEG:
{
  "raw_label": "G2",
  "semantic_label": "medium_diameter_grasp",
  "confidence": 0.7448
}

EMG:
{
  "raw_label": 0,
  "semantic_label": "large_diameter_grasp",
  "confidence": 0.9867
}

VISION:
{
  "hands": [
    {
      "handedness": "right",
      "contact_state": "no_contact",
      "primary_object": "unknown",
      "hand_object_relation": "unknown",
      "grasp_type": "unknown"
    }
  ]
}

DETERMINISTIC FUSION:
{
  "neural_gesture_agreement": false,
  "average_neural_confidence": 0.86575,
  "vision_object_present": false,
  "vision_active_interaction": false,
  "vision_object_matches_instruction": false,
  "decision": "WAIT"
}

LLM REASONING:
{
  "inferred_intention": "grasp",
  "requested_action": "Pick up the cup",
  "eeg_consistent": true,
  "emg_consistent": true,
  "vision_consistent": false,
  "overall_decision": "WAIT",
  "confidence": 0.75,
  "reason": "Both EEG and EMG indicate a grasping gesture, but the visi

# MORE TESTS

In [9]:
# T2) different object
result_test_2 = run_multimodal_pipeline(
    task_instruction="Pick up the bottle.",
    image_path=IMAGE_PATH,
    eeg_path=EEG_PATH,
    emg_path=EMG_PATH,
    run_llm=True
)

display_clean_result(result_test_2)

MULTIMODAL INFERENCE RESULT

TASK:
Pick up the bottle.

EEG:
{
  "raw_label": "G2",
  "semantic_label": "medium_diameter_grasp",
  "confidence": 0.7448
}

EMG:
{
  "raw_label": 0,
  "semantic_label": "large_diameter_grasp",
  "confidence": 0.9867
}

VISION:
{
  "hands": [
    {
      "handedness": "right",
      "contact_state": "no_contact",
      "primary_object": "unknown",
      "hand_object_relation": "unknown",
      "grasp_type": "unknown"
    }
  ]
}

DETERMINISTIC FUSION:
{
  "neural_gesture_agreement": false,
  "average_neural_confidence": 0.86575,
  "vision_object_present": false,
  "vision_active_interaction": false,
  "vision_object_matches_instruction": false,
  "decision": "WAIT"
}

LLM REASONING:
{
  "inferred_intention": "grasp_bottle",
  "requested_action": "pick_up_the_bottle",
  "eeg_consistent": true,
  "emg_consistent": true,
  "vision_consistent": false,
  "overall_decision": "WAIT",
  "confidence": 0.75,
  "reason": "The EEG and EMG gesture predictions are consi

In [10]:

#T3) Open-hand action
result_test_3 = run_multimodal_pipeline(
    task_instruction="Release the object.",
    image_path=IMAGE_PATH,
    eeg_path=EEG_PATH,
    emg_path=EMG_PATH,
    run_llm=True
)

display_clean_result(result_test_3)

MULTIMODAL INFERENCE RESULT

TASK:
Release the object.

EEG:
{
  "raw_label": "G2",
  "semantic_label": "medium_diameter_grasp",
  "confidence": 0.7448
}

EMG:
{
  "raw_label": 0,
  "semantic_label": "large_diameter_grasp",
  "confidence": 0.9867
}

VISION:
{
  "hands": [
    {
      "handedness": "right",
      "contact_state": "no_contact",
      "primary_object": "unknown",
      "hand_object_relation": "unknown",
      "grasp_type": "unknown"
    }
  ]
}

DETERMINISTIC FUSION:
{
  "neural_gesture_agreement": false,
  "average_neural_confidence": 0.86575,
  "vision_object_present": false,
  "vision_active_interaction": false,
  "vision_object_matches_instruction": false,
  "decision": "WAIT"
}

LLM REASONING:
{
  "inferred_intention": "grasp_or_attempt_to_grasp",
  "requested_action": "release",
  "eeg_consistent": false,
  "emg_consistent": false,
  "vision_consistent": false,
  "overall_decision": "REJECT",
  "confidence": 0.95,
  "reason": "The user requested a release action, bu